## Generación de suciedad de forma realista

Este notebook inyecta nulos, outliers y duplicados en el dataset original. 

> **Nota sobre Rutas**: El script detecta automáticamente si se está ejecutando desde la raíz del proyecto o desde la carpeta `notebooks/` para evitar errores de archivo no encontrado.

In [ ]:
import pandas as pd
import numpy as np
import os

# Gestión de rutas inteligente
cwd = os.getcwd()
if cwd.endswith('notebooks'):
    base_path = os.path.join('..', 'data')
else:
    base_path = 'data'

path_original = os.path.join(base_path, 'dataset_ORIGINAL.csv')
path_sucio = os.path.join(base_path, 'dataset_DIRTY.csv')

# 1. Cargamos el dataset original
print(f"Intentando cargar: {path_original}")
df = pd.read_csv(path_original)
print("Dataset original cargado con éxito.")

# 2. DISTRIBUCIÓN DE NULOS
df.loc[df.sample(frac=0.02).index, 'Previous_GPA'] = np.nan
df.loc[df.sample(frac=0.01).index, 'Weekly_Study_Hours'] = np.nan
df.loc[df.sample(frac=0.01).index, 'Attendance_Rate'] = np.nan

# 3. INYECCIÓN DE OUTLIERS
indices_age = df.sample(n=100).index
df.loc[indices_age[:50], 'Age'] = 0
df.loc[indices_age[50:], 'Age'] = 150
df.loc[df.sample(n=50).index, 'Previous_GPA'] = -5.0
df.loc[df.sample(n=30).index, 'Weekly_Study_Hours'] = 500

# 4. DUPLICADOS
filas_a_duplicar = df.sample(n=100)
df = pd.concat([df, filas_a_duplicar], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)

# 5. GUARDAMOS EL DESASTRE
df.to_csv(path_sucio, index=False)
print(f"¡Dataset SUCIO generado en: {path_sucio}!")


## Verificación de Auditoría y Muestras de "Basura"

En esta celda no solo contamos los errores, sino que mostramos ejemplos reales de las filas afectadas para validar la calidad del proceso de ensuciado.

In [ ]:
df_sucio = pd.read_csv(path_sucio)
print("=== REPORTE DE AUDITORÍA DETALLADO ===\n")

# Configuramos pandas para que no nos esconda columnas
pd.set_option('display.max_columns', None)

# Columnas de interés para la auditoría
cols_audit = ['Student_ID', 'Age', 'Previous_GPA', 'Weekly_Study_Hours', 'Attendance_Rate']

# 1. Chequeo de Nulos con muestra
print("--- 1. NULOS ---")
nulos_totales = df_sucio.isna().sum()
print(nulos_totales[nulos_totales > 0])
print("\nEjemplo de filas con valores nulos (fijate en Previous_GPA, Weekly_Study_Hours o Attendance_Rate):")
display(df_sucio[df_sucio.isna().any(axis=1)][cols_audit].head(5))

# 2. Chequeo de Outliers con muestra
print("\n--- 2. OUTLIERS ---")
out_age = df_sucio[(df_sucio['Age'] <= 0) | (df_sucio['Age'] >= 120)]
out_gpa = df_sucio[df_sucio['Previous_GPA'] < 0]
out_hours = df_sucio[df_sucio['Weekly_Study_Hours'] > 168]

print(f"Edades imposibles (<=0 o >=120): {len(out_age)}")
print(f"GPAs negativos: {len(out_gpa)}")
print(f"Horas de estudio imposibles (>168): {len(out_hours)}")

print("\nEjemplo de outliers detectados (combinados):")
all_outliers = pd.concat([out_age.head(2), out_gpa.head(2), out_hours.head(2)])
display(all_outliers[cols_audit])

# 3. Chequeo de Duplicados
dups_mask = df_sucio.duplicated(keep=False)
dups_count = df_sucio.duplicated().sum()
print(f"\n--- 3. DUPLICADOS ---\nFilas duplicadas exactas: {dups_count}")

if dups_count > 0:
    print("\nMuestra de filas duplicadas (viendo pares para comparar):")
    display(df_sucio[dups_mask].sort_values(by='Student_ID')[cols_audit].head(6))
